## 🧠 TinyStories Language Model – A 15M Parameter GPT from Scratch

This project is an educational implementation of a **small-scale GPT-style language model**, built entirely from scratch using **PyTorch** and trained on the **TinyStories** dataset introduced by Apple.

The goal is to understand the internal workings of large language models (LLMs) by developing a **15 million parameter Transformer**, taking architectural inspiration from **GPT-3**. The TinyStories dataset is ideal for this purpose due to its simplicity, small size, and well-formed language structure.

Target:
A model that make stories for children and those stories should produce coherent text, stories should convey some meaning and the model should be able to understand correct grammer for some far.


### 📚 References
- 📄 [TinyStories: How Small Can Language Models Be and Still Speak Coherently? (Apple, 2023)](https://arxiv.org/abs/2305.07759)
- 📄 [Language Models are Few-Shot Learners (GPT-3 paper)](https://arxiv.org/abs/2005.14165)


### PART-1 : EXPLORING THE DATASET

📚 Dataset: TinyStories

The TinyStories dataset is a collection of short, synthetically generated children's stories created using GPT-3.5 and filtered for coherence, simplicity, and grammatical correctness. It provides a compact, high-quality corpus ideal for training small language models in a low-compute environment. Each story is typically 200–300 tokens long and written in simple, well-structured English.

In [13]:
# Tiny Stories dataset can be found in huggingface
# Those datasets can imported using dataset lib

from datasets import load_dataset

ds = load_dataset('roneneldan/TinyStories')

In [14]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})


In [15]:
print(ds['train'].features)

{'text': Value(dtype='string', id=None)}


In [16]:
for i in range(3):
    print(ds['train'][i])


{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}
{'text': 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had man

### PART-2 : DATA PREPROCESSING

## 🔤 Byte Pair Encoding (BPE) – Quick Reference

### 🔧 What is BPE?
Byte Pair Encoding (BPE) is a **subword tokenization** method that sits between word-level and character-level encoding. It merges the most frequent pairs of characters or subwords to form new tokens, allowing the model to:
- Keep vocabulary small,
- Handle unknown words effectively,
- Shorten input sequences compared to character-level tokenization.

---

### 🚧 Why Use BPE?
- ✅ Compact vocabulary → faster and more memory-efficient training
- ✅ Robust to rare and unseen words
- ✅ Shorter token sequences than character-level
- ✅ Proven in LLMs like GPT-2 and GPT-3

---

### ❌ Why Not Word-Level or Character-Level?

#### ❌ Word-Level Encoding
- 🧨 **Huge vocabulary** → requires more memory and larger embedding layers
- 📉 **Fails on rare/misspelled/compound words** (OOV problem)
- 🚫 **Not scalable** for LLM training

#### ❌ Character-Level Encoding
- 🧪 **Simpler to implement**, but...
- 🐢 **Leads to long token sequences** → slower training and inference
- ❌ **Harder for model to learn meaningful representations**

---

### ⚙️ How BPE Works
1. Start with character-level tokens (e.g., `s t o r y </w>`).
2. Count the frequency of all adjacent token pairs (e.g., `t h`, `h e`, etc.).
3. Merge the most frequent pair into a new subword token (e.g., `th`, `he`).
4. Repeat until desired vocabulary size is reached (e.g., 5,000–8,000 tokens).

---

### 🧠 Example
Input sentence:  
`the ship is sinking`

Initial tokens:  
`t h e</w>` `s h i p</w>` `i s</w>` `s i n k i n g</w>`

After frequent merges:  
**`the` `ship` `is` `sink` `ing`**

---

### 🛠️ Tooling
- 📦 Using tiktoken library (pip install tiktoken)

---

### 📌 Practical Setup for This Project
- 🧠 Train a custom BPE tokenizer on TinyStories
- 🧰 Target vocab size: **~5,000–8,000**
- 📏 Max sequence length: **128–256 tokens**

---


In [29]:
import tiktoken
import os
import numpy as np
from tqdm.auto import tqdm

# Creates an encoder instance for the GPT-2 tokenizer.
enc = tiktoken.get_encoding("gpt2")

# Encodes the text into token IDs using encode_ordinary
# (which encodes text without special tokens).
def process(example):
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    ids = enc.encode_ordinary(example['text'])  # encode_ordinary ignores special tokens
    out = {'ids': ids, 'len': len(ids)}
    return out


In [30]:
sample = {"text": "Once upon a time in a tiny land."}
print(process(sample))

{'ids': [7454, 2402, 257, 640, 287, 257, 7009, 1956, 13], 'len': 9}


| Token ID | Approximate Text | Notes                           |
|----------|------------------|--------------------------------|
| 7454     | Once             | Full word or common subword    |
| 2402     |  upon            | Leading space included (` `)   |
| 257      |  a               | Leading space                  |
| 640      |  time            | Leading space                  |
| 287      |  in              | Leading space                  |
| 257      |  a               | Leading space                  |
| 7009     |  tiny            | Leading space                  |
| 1956     |  land            | Leading space                  |
| 13       | .                | Punctuation token              |


In [31]:
if not os.path.exists("train.bin"):
    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="tokenizing the splits",
        num_proc=4,
        )
    # concatenate all the ids in each dataset into one large file we can use for training
    for split, dset in tokenized.items():
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16 # (can do since enc.max_token_value == 50256 is < 2**16)
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))
        total_batches = 1024

        idx = 0
        for batch_idx in tqdm(range(total_batches), desc=f'writing {filename}'):
            # Batch together samples for faster write
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            # Write into mmap
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()

tokenizing the splits (num_proc=4):   0%|          | 0/2119719 [00:00<?, ? examples/s]

tokenizing the splits (num_proc=4):   0%|          | 0/21990 [00:00<?, ? examples/s]

writing train.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

writing validation.bin:   0%|          | 0/1024 [00:00<?, ?it/s]

###  PART-3 : CREATING INPUT AND TARGET BATCHES

# Input and Output (Targets) for the Language Model

## Causal Language Modeling (CLM)

For GPT-style models, the task is to **predict the next token** in a sequence.

- Inputs are tokens excluding the last token.
- Targets are tokens shifted by one (the next token).
- Model learns to predict each next token autoregressively.

---

## Input / Target Setup

Given a tokenized sequence:


We create:

- **Inputs (`x`)** = `tokens[:-1]`  
- **Targets (`y`)** = `tokens[1:]`

This means the model learns to predict the next token at each position.

---

## Example

For tokenized text:  
`[502, 309, 356, 1143, 13]`

- Inputs: `[502, 309, 356, 1143]`  
- Targets: `[309, 356, 1143, 13]`

---

## Training Data Batching

When training, split the token stream into fixed-length chunks (e.g., block size = 128):

| Input (x)               | Target (y)              |
|------------------------|-------------------------|
| `[t0, t1, ..., t127]`  | `[t1, t2, ..., t128]`   |
| `[t128, ..., t255]`    | `[t129, ..., t256]`     |
| ...                    | ...                     |








## Batch Size and Context Size (Sequence Length)

### Batch Size

- **Definition:** The number of separate sequences processed together in one training step.
- **Purpose:**  
  - Enables parallel processing on GPUs/TPUs for faster training.  
  - Larger batch sizes can improve training stability but require more memory.
- **Example:**  
  If `batch_size = 32`, the model processes 32 sequences at once.

---

### Context Size (Sequence Length)

- **Definition:** The number of tokens in each input sequence fed into the model.
- **Purpose:**  
  - Determines how many previous tokens the model "remembers" to predict the next token.  
  - Also called **window size** or **block size**.
- **Example:**  
  If `context_size = 128`, each input sequence is 128 tokens long.

---

### How They Work Together

- Each training step inputs a batch of sequences shaped:  
  `[batch_size, context_size]`
- The model predicts the next token for each position in each sequence.

---

### Summary

| Term          | Meaning                                  |
|---------------|------------------------------------------|
| Batch Size    | Number of sequences processed at once    |
| Context Size  | Number of tokens per input sequence       |



In [32]:
def get_batch(split):
    # We recreate np.memmap every batch to avoid a memory leak, as per
    # https://stackoverflow.com/questions/45132940/numpy-memmap-memory-usage-want-to-iterate-once/61472122#61472122
    # Uses np.memmap to memory-map the binary file without fully loading it into RAM.
    # dtype=np.uint16 means each token is stored as a 16-bit unsigned integer
    # mode='r' indicates read-only access.
    # This avoids loading the entire file, which is helpful for large datasets.
    if split == 'train':
        data = np.memmap('train.bin', dtype=np.uint16, mode='r')
    else:
        data = np.memmap('validation.bin', dtype=np.uint16, mode='r')
    
    # Samples batch_size random starting indices (ix) from the dataset, making sure not to exceed the file length.
    # block_size is the number of tokens in each training sequence
    # len(data) - block_size ensures we don’t go out of bounds.
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # For each index i in ix, two sequences are extracted:
    # x: input sequence of length block_size, starting at i
    # y: target sequence of length block_size, starting at i+1 (i.e., next token after each input)
    # Each sequence is cast to np.int64, converted to a PyTorch tensor with torch.from_numpy, and stacked into a batch.
    x = torch.stack([torch.from_numpy((data[i:i+block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+block_size]).astype(np.int64)) for i in ix])
    
    # If running on GPU (cuda), pin the memory (using .pin_memory()), enabling faster and asynchronous .to(device) transfer.
    # If on CPU, just move to device directly.
    if device_type == 'cuda':
        # pin arrays x,y, which allows us to move them to GPU asynchronously (non_blocking=True)
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

### PART-4 DEFINING THE SLM MODEL ARCHITECTURE

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
import numpy as np
from tqdm.auto import tqdm
from contextlib import nullcontext
import os

#### Layer Normalization
- Applies **layer normalization** using PyTorch's functional API `F.layer_norm`.
- It normalizes the last dimension of the tensor `x`:
  - Subtracts mean and divides by standard deviation (like z-score).
  - Then applies `weight` and (optionally) `bias`.
- `1e-5`: small epsilon added for numerical stability.

This class behaves similarly to PyTorch’s built-in `nn.LayerNorm`, but gives you flexibility to control whether a bias term is used.


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

In [ ]:
# You can then pass config to your model and its subcomponents to build everything based on these shared settings.
@dataclass
class GPTConfig:
    block_size: int # Maximum context length (number of tokens the model sees at once)
    vocab_size: int # Size of the tokenizer vocabulary (number of unique token IDs)
    n_layer: int # Number of Transformer blocks (depth of the model)
    n_head: int # Number of attention heads in multi-head attention
    n_embd: int # Dimensionality of token embeddings and model layers
    dropout: float = 0.0 # Dropout rate for regularization (optional, default 0)
    bias: bool = True # Whether to use bias in linear layers (optional, default True)

### 🔍 Attention Mechanism & Context Vectors

---

### 🎯 What Is the Attention Mechanism?

The **attention mechanism** allows a model to dynamically focus on different parts of the input sequence when processing a token.

It answers the question:  
> **"Which other tokens in the input should I pay attention to when processing this one?"**

#### ⚙️ How It Works (Self-Attention):
For each token:
1. **Query (Q)** (What am I looking for?), **Key (K)** (What do I offer to match?), and **Value (V)**(What information do I carry?) vectors are computed.
2. The dot product of Q with all Ks gives attention scores.
3. Softmax turns scores into attention weights (how much attention to pay to each token).
4. A **weighted sum of the Value vectors** is computed → this is the **context vector**.

> 📌 **Context Vector** = What the token "knows" after attending to the rest of the sequence.

---

### 🧠 What Is a Context Vector?

A **context vector** is a token's representation **after it has attended to all other tokens**.  
It captures the **meaning of the token in context**.

#### Example:
In:
> *"She went to the **bank** to deposit money."*

The word "**bank**" attends to "deposit" and "money", so its context vector represents the **financial meaning**.

---

### 💡 Why Is Attention Important?

- Models understand **relationships between words**, regardless of position.
- Attention enables **long-range dependencies** (e.g., subject at beginning, verb at end).
- Better than fixed-size windows (like RNNs or CNNs used before Transformers).

---

### 🧬 Summary Table

| Concept           | Description |
|------------------|-------------|
| **Attention**     | Mechanism to weigh importance of other tokens |
| **Self-Attention**| Each token attends to all tokens (including itself) |
| **Context Vector**| Weighted sum of other tokens' values, representing context-aware meaning |
| **Why It Matters**| Captures meaning based on entire sequence, enabling powerful understanding and generation |

---

### 🧠 Self-Attention Explained (with 4 Tokens × 768-Dim Vectors)

This explains how the **self-attention mechanism** works in Transformer models (like GPT), using a simplified example with:

- **Sequence length**: 4 tokens  
- **Embedding size**: 768 dimensions  
- **Projection matrices**: 768 × 768 for Query, Key, and Value

---

#### 🔢 Step 1: Input Representation

Suppose we have 4 tokens in a sequence. Each token is represented by a 768-dimensional vector.

So, the **input matrix** `X` is:X ∈ ℝ⁴ˣ⁷⁶⁸


| Token | Embedding Vector (Dim: 768) |
|-------|-----------------------------|
| T₀    | x₀                          |
| T₁    | x₁                          |
| T₂    | x₂                          |
| T₃    | x₃                          |

---

#### 🧮 Step 2: Create Q, K, V Projections

We apply **3 learnable weight matrices** to compute:

- **Query (Q)**: What this token is looking for  
- **Key (K)**: What this token contains  
- **Value (V)**: The information held by the token

Each projection uses a weight matrix:
- `W_Q ∈ ℝ⁷⁶⁸ˣ⁷⁶⁸`
- `W_K ∈ ℝ⁷⁶⁸ˣ⁷⁶⁸`
- `W_V ∈ ℝ⁷⁶⁸ˣ⁷⁶⁸`

Then compute:
Q = X × W_Q → ℝ⁴ˣ⁷⁶⁸
K = X × W_K → ℝ⁴ˣ⁷⁶⁸
V = X × W_V → ℝ⁴ˣ⁷⁶⁸


---

#### 📐 Step 3: Attention Scores (Dot Product QKᵀ)

We calculate how much attention each token should pay to every other token using:

Attention_Scores = Q × Kᵀ → ℝ⁴ˣ⁴


Each cell `(i, j)` in this 4×4 matrix tells how much token `i` attends to token `j`.

Here the there shouldn't be attntion scores related to the token after some particular token. Therefoer those scores should be zero - Cousal Attention. (This mean elements above the diagonal should be zero)

It ensures that each token in a sequence can only attend to itself and earlier tokens (causality),
---

#### 🧊 Step 4: Scale the Scores

To stabilize gradients, scale scores by the square root of the dimension (768):

Scaled_Scores = Attention_Scores / sqrt(768)

---

#### 🔁 Step 5: Softmax Over Rows

Apply **softmax** across each row (for each query):

Attention_Weights = softmax(Scaled_Scores, dim=1) → ℝ⁴ˣ⁴

Each row now contains weights that sum to 1 — determining how much each token attends to others.

---

#### 🧠 Step 6: Compute Context Vectors

We use the attention weights to compute a **weighted sum** of values (V):

Context = Attention_Weights × V → ℝ⁴ˣ⁷⁶⁸


Now each token has a **new representation** (called a **context vector**) that contains information from the entire sequence.

---

#### ✅ Final Output

The `Context` matrix (shape: `4×768`) becomes the output of the self-attention block for this sequence.

It contains context-aware token embeddings — each token now "knows" about the others in the sequence.

This output is then passed to the next layer (e.g., another attention block or a feedforward network).

---

#### 🔄 Notes on Multi-Head Attention

In real Transformer models, this process is done **multiple times in parallel** with smaller projection dimensions (e.g., 768 → 64 × 12 heads). Each head captures different types of relationships, and their outputs are concatenated and linearly transformed.


In [ ]:
# This will take the input embedding matrix and covert it in to context vector matrix
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # n_embd - embedding size (e.g., 768)
        # n_head - number of attention heads (e.g., 12)
        # Sanity Check - Ensures the embedding dimension is divisible by the number of heads
        assert config.n_embd % config.n_head == 0
        
        # Projects the input into query, key, value vectors in one go:
        # Takes input shape [B, T, C] → outputs [B, T, 3C]
        # [B,T, C] = [batch_size, sequence_length, embedding_dim]
        # Then it's split into Q, K, V.
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # Final projection after attention is applied (combines all heads).
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        
        # Dropout for regularization.
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        
        # Flash Attention Support (Optional Speedup)
        # Checks if PyTorch has efficient Flash Attention support
        self.flash = hasattr(F, 'scaled_dot_product_attention')
        if not self.flash:
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                       .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size()
        # Splits (B, T, 3C) into q, k, v, each of shape (B, T, C)
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        
        # Reshape and transpose to split into multiple heads.
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.attn_dropout.p if self.training else 0.0, is_causal=True)
        else:
            # Calculate Attention Scores
            # This computes similarity between each token and all previous tokens.
            # If token A's query matches token B's key → high score
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            # Apply Causal Mask
            # This will make all the elements above the diagonal in the attention matrix negative infinity,
            # So when we apply softmax, they become zero.
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            # Softmax + Dropout
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            # Weighted Sum of Values
            # y = att @ v is equal y = torch.matmul(att, v)
            # just matrix multiplication
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

### 🔧 MLP Module in Transformer (Feed-Forward Network)

#### 🧠 Purpose

The `MLP` (Multi-Layer Perceptron) module is a **feed-forward network** applied to each token **independently** after the self-attention mechanism. It increases the model's capacity to learn complex transformations for each token representation.
(This basically expand the vector into a higher dimension to capture more details learn more complex patterns and again bring it to the original dimension)

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        # first linear layer expands the input from n_embd to 4 * n_embd
        # The idea: more capacity for learning complex patterns
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        
        # Gaussian Error Linear Unit (GELU) activation function
        # It applies a non-linear transformation to the input, allowing the model to learn complex patterns
        self.gelu = nn.GELU()
        
        # second linear layer projects the 4× expanded representation back down to original size
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        # Randomly zeroes some outputs during training for regularization
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

#### 🔧 Transformer Block (`Block` Class)

The `Block` class defines a single **Transformer decoder block**, the core unit of models like GPT. It contains:

- Layer Normalization (`LayerNorm`)
- Causal Self-Attention (`CausalSelfAttention`)
- Feed-forward Neural Network (`MLP`)
- Residual Connections

In [ ]:
# This is the main Transformer block that combines self-attention and MLP
# This uses all the components we defined above
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        # LayerNorm before attention and MLP
        # Helps stabilize and accelerate training.
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        
        # This is the masked multi-head self-attention layer.
        # Output: context-aware version of each token.
        self.attn = CausalSelfAttention(config)
        
        # Another normalization before the feedforward network (MLP).
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        
        # A 2-layer fully connected neural network which adds non-linearity.
        self.mlp = MLP(config)
    def forward(self, x):
        # Residual connection around self-attention and MLP
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

### 🔁 Residual Connections in Transformers

Residual connections are a fundamental concept in modern deep learning models, especially **transformers** like GPT. They allow models to train deeper networks efficiently and effectively.

### 📌 What is a Residual Connection?

A **residual connection** adds the input of a layer directly to its output:

output = x + F(x)
Final version = original + all improvements (residuals).

### 🎯 Why Use Residual Connections?
✅ 1. Better Gradient Flow
- In deep networks, gradients can vanish as they are backpropagated.
- Residual connections directly pass gradients backward through the skip path.

✅ 2. Preserve Input Information
- Not every layer needs to completely transform the input.
- If the transformation isn't helpful, the model can learn to do nothing

Without residuals:
- Each layer would fully overwrite the previous layer's result.

With residuals:
- Each layer adds its contribution, preserving previous knowledge and allowing flexibility.

### 🧠 GPT Class Overview

This class implements a GPT (Generative Pretrained Transformer) language model using PyTorch. It mimics the architecture of GPT-2 in a simplified, readable form.

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.n_embd), # word/token embeddings
            wpe=nn.Embedding(config.block_size, config.n_embd), # positional embeddings
            drop=nn.Dropout(config.dropout), # dropout for regularization
            h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]), # list of Transformer blocks
            ln_f=LayerNorm(config.n_embd, config.bias), # final layer normalization
        ))

        # Language model head (final linear layer)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # Weight tying (output weights = embedding weights)
        self.transformer.wte.weight = self.lm_head.weight

        # Initialize weights
        # This applies the _init_weights function to all modules in the model.
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer))

    # Initializes weights for Linear and Embedding layers using a normal distribution (mean 0, std 0.02).
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        # 1. Input shape checks:
        b, t = idx.size()
        assert t <= self.config.block_size
        # 2. Create position indices:
        # This creates a tensor of position indices from 0 to t-1.
        pos = torch.arange(0, t, dtype=torch.long, device=device)

        # 3. Embed tokens and positions:
        tok_emb = self.transformer.wte(idx) # (B, T, C)
        pos_emb = self.transformer.wpe(pos) # (T, C)
        x = self.transformer.drop(tok_emb + pos_emb)
        
        # 4. Pass through Transformer blocks:
        for block in self.transformer.h:
            x = block(x)
        # 5. Final Layer Normalization:
        x = self.transformer.ln_f(x)

        # 6.1 If targets is provided, compute the compute cross-entropy loss:
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
            return logits, loss
        else:
            # 6.2 Output projection to vocabulary:
            # If targets is None, return only the last time step logits (used for sampling):
            logits = self.lm_head(x[:, [-1], :])
            return logits, None
        
    # @torch.no_grad() will Tell PyTorch not to track gradients (inference mode)
    # Saves memory & improves speed
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        This function:
            - Predicts the next tokens one by one
            - Takes an initial token sequence idx of shape (B, T)
            - Returns a sequence of shape (B, T + max_new_tokens)
        The key idea:
            - Autoregressive generation, i.e., "generate one token, then feed it back in to generate the next."
        """
        # Each iteration generates one token and appends it to idx
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

| Parameter   | Value  | Description |
|-------------|--------|-------------|
| `vocab_size` | 50257 | Vocabulary size used by the tokenizer (GPT-2's vocabulary size). Defines how many unique tokens the model can understand and predict. |
| `block_size` | 128   | Context window size. The maximum number of tokens the model processes in one forward pass. Smaller values reduce memory usage and training time. |
| `n_layer`    | 6     | Number of Transformer blocks. Controls model depth. |
| `n_head`     | 6     | Number of attention heads per block. Must divide `n_embd` evenly. More heads allow better parallel representation learning. |
| `n_embd`     | 384   | Embedding size (i.e., the dimensionality of token vectors). Determines the expressiveness and capacity of the model. |
| `dropout`    | 0.1   | Dropout rate to help prevent overfitting. |
| `bias`       | True  | Whether to include bias terms in linear layers. |


In [ ]:
# This creates a GPTConfig object — a container that holds all the model's hyperparameters.
config = GPTConfig(
    vocab_size=50257,   # Number of unique tokens in the vocabulary
    block_size=128,     # Max number of tokens the model sees at once
    n_layer=6,          # Number of transformer blocks
    n_head=6,           # Number of attention heads in each block
    n_embd=384,         # Size of token embeddings and hidden states
    dropout=0.1,        # Dropout rate
    bias=True           # Whether linear layers use bias terms.
)

model = GPT(config)

### PART-5 LOSS ESTIMATION

In [ ]:
def estimate_loss(model):
    """ 
        This is used to evaluate the average loss of the GPT 
        model on both the training and validation sets, 
        without updating weights.
    """
    out = {}
    model.eval()
    with torch.inference_mode():
        for split in ['train', 'val']:
            losses = torch.zeros(eval_iters)
            for k in range(eval_iters):
                X, Y = get_batch(split)
                with ctx:
                    logits, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean()
    model.train()
    return out

### PART 6: SLM TRAINING CONFIGURATION-1

In [ ]:
# Training Config
import torch
from contextlib import nullcontext

learning_rate = 1e-4    # Lower learning rate for stable training
max_iters = 20000       # Total number of training iterations
warmup_steps = 1000     # Gradual ramp-up of learning rate in the first few steps
min_lr = 5e-4           # Minimum learning rate after cosine decay (usually this would be *lower* than initial LR!)
eval_iters = 500        # Number of batches to average over when estimating loss
batch_size = 32         # More samples = more stable gradients
block_size = 128        # How many tokens the model can see at once

# this breaks a large batch into smaller chunks to fit in memory
gradient_accumulation_steps = 32 # Number of steps to accumulate gradients before updating

# Device Configuration
device =  "cuda" if torch.cuda.is_available() else "cpu"
device_type = 'cuda' if 'cuda' in device else 'cpu' # for later use in torch.autocast
# note: float16 data type will automatically use a GradScaler

# How to use autocast https://wandb.ai/wandb_fc/tips/reports/How-To-Use-Autocast-in-PyTorch--VmlldzoyMTk4NTky
#dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
dtype = 'bfloat16' if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'float16' # 'float32', 'bfloat16', or 'float16', the latter will auto implement a GradScaler
ptdtype = {'float32': torch.float32, 'bfloat16': torch.bfloat16, 'float16': torch.float16}[dtype]

ctx = nullcontext() if device_type == 'cpu' else torch.amp.autocast(device_type=device_type, dtype=ptdtype)

# Ensures all tensors are created on the correct device
torch.set_default_device(device)
torch.manual_seed(42)

### PART 7: SLM TRAINING CONFIGURATION-2

🧠 **Why Use Mixed Precision (AMP)?**

Normally, neural networks use `float32`, which is accurate but slow and memory-intensive. With **Automatic Mixed Precision (AMP)**, most operations use `float16`, which is faster and uses less memory, while sensitive operations (like loss computation and weight updates) stay in `float32` to preserve accuracy.

✅ This results in **faster training**, **lower memory usage**, and the ability to train **larger models**.

🚨 However, `float16` has a limited dynamic range — small gradients can vanish (underflow), which can stop training.

🛟 **Solution: `GradScaler`**
- Scales up the loss before backward pass
- Keeps gradients large enough to avoid underflow
- Unscales before optimizer step
- Skips updates if gradients are `inf` or `NaN`
- Automatically adjusts scaling over time

AMP + `GradScaler` = safe and efficient mixed precision training on GPU.


In [ ]:
from torch.optim.lr_scheduler import LinearLR,SequentialLR, CosineAnnealingLR

"""
    This code configures the optimizer, learning rate scheduler, 
    and mixed-precision training scaler for training your GPT model 
    using PyTorch.
"""

# The AdamW optimizer, which is a better version of Adam for training 
# transformers because it decouples weight decay from the gradient update.
optimizer =  torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1, eps=1e-9) #weight decay for regularization

# This controls how the learning rate evolves over time:

# 1. Linear warmup: gradually increases the learning rate from 0 to the initial learning rate over a specified number of steps (warmup_steps).
scheduler_warmup = LinearLR(optimizer, total_iters = warmup_steps) #Implement linear warmup
# 2. Cosine decay: after warmup, the learning rate decays following a cosine function until it reaches a minimum value (min_lr) at the end of training.
scheduler_decay = CosineAnnealingLR(optimizer,T_max = max_iters - warmup_steps, eta_min = min_lr) #Implement lr decay
# This combines both warmup and decay into a single scheduler.
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[warmup_steps]) #Switching from warmup to decay

# https://stackoverflow.com/questions/72534859/is-gradscaler-necessary-with-mixed-precision-training-with-pytorch
scaler = torch.cuda.amp.GradScaler('cuda',enabled=(dtype == 'float16'))
# AMP = Automatic Mixed Precision — it helps train faster with less memory on GPUs by using lower precision where safe.

### PART - 8 PRE-TRAINING

In [ ]:
best_val_loss = float('inf')
best_model_params_path = "best_model_params.pt"
train_loss_list, validation_loss_list = [], []

# Ensure model is on the correct device
model = model.to(device)

# Main training loop
for epoch in tqdm(range(max_iters)):
    if epoch % eval_iters == 0 and epoch != 0:
        # Ensure estimate_loss uses the correct device
        losses = estimate_loss(model)
        print(f"Epoch {epoch}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"The current learning rate: {optimizer.param_groups[0]['lr']:.5f}")
        train_loss_list += [losses['train']]
        validation_loss_list += [losses['val']]
        
        # Save the model parameters if validation loss improves
        if losses['val'] < best_val_loss:
            best_val_loss = losses['val']
            torch.save(model.state_dict(), best_model_params_path)

    # Get a batch of training data
    X, y = get_batch("train")
    X, y = X.to(device), y.to(device)

    # forward + Backward with AMP and Gradient Accumulation
    with ctx:
        logits, loss = model(X, y)
        loss = loss / gradient_accumulation_steps
        scaler.scale(loss).backward()

    # Perform Optimizer Step Every N Iterations
    if ((epoch + 1) % gradient_accumulation_steps == 0) or (epoch + 1 == max_iters):
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
    # Update the learning rate scheduler
    scheduler.step()

### PART - 9 SLM LOSS 

In [ ]:
import matplotlib.pyplot as plt
train_loss_list_converted = [i.cpu().detach() for i in train_loss_list]
validation_loss_list_converted = [i.cpu().detach() for i in validation_loss_list]

plt.plot(train_loss_list_converted, 'g', label='train_loss')
plt.plot(validation_loss_list_converted, 'r', label='validation_loss')
plt.xlabel("Steps - Every 100 epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

### PART - 10 RUNNING INFERENCE

In [ ]:
#Load the model
model = GPT(config)  # re-create the model with same config
device =  "cuda" if torch.cuda.is_available() else "cpu"
best_model_params_path = "best_model_params.pt"
model.load_state_dict(torch.load(best_model_params_path, map_location=torch.device(device))) # load best model states


In [ ]:
sentence = "Once upon a time there was a pumpkin."
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))

In [ ]:
sentence = "A little girl went to the woods"
context = (torch.tensor(enc.encode_ordinary(sentence)).unsqueeze(dim = 0))
y = model.generate(context, 200)
print(enc.decode(y.squeeze().tolist()))